# AI Compliance Validator (Insurance Claim Auditing)
End-to-end AI Compliance Validation Agent using RAG (ChromaDB), TXT/PDF ingestion (PyMuPDF), strict JSON prompting, PydanticAI, and a local vLLM OpenAI-compatible endpoint for AMD GPU inference.

Run this notebook cell-by-cell.

## 1) Environment Setup

In [ ]:
import sys
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
)
logger = logging.getLogger('ai_compliance_validator')

def pip_install(packages):
    """Install packages via pip (best-effort, idempotent)."""
    import subprocess
    for pkg in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = [
    'pydantic>=2.0',
    'pydantic-ai>=0.0.18',
    'requests',
    'chromadb',
    'sentence-transformers',
    'pymupdf',
    'fastapi',
    'uvicorn',
    'matplotlib',
    'pandas'
]

# Install only if key imports fail
try:
    import chromadb  # noqa: F401
    import pydantic_ai  # noqa: F401
    from sentence_transformers import SentenceTransformer  # noqa: F401
    import fitz  # noqa: F401
except Exception:
    logger.info('Installing missing dependencies...')
    pip_install(required)

def check_gpu():
    """Best-effort GPU check using torch (works for many backends)."""
    try:
        import torch
        if torch.cuda.is_available():
            return {
                'cuda_available': True,
                'device_count': torch.cuda.device_count(),
                'device_name': torch.cuda.get_device_name(0),
            }
        return {'cuda_available': False}
    except Exception as e:
        return {'cuda_available': False, 'error': str(e)}

gpu_info = check_gpu()
logger.info(f'GPU info: {gpu_info}')
gpu_info

## 2) Configuration

In [ ]:
import os
import json
import re
from datetime import datetime
from typing import Any, Optional

from pathlib import Path

AMD_BASE_URL = "http://localhost:8000/v1"
AMD_API_KEY = "abc-123"
MODEL_NAME = "Qwen/Qwen3-4B"

REPO_ROOT = Path(os.getcwd())
RULEBOOK_PATH = REPO_ROOT / 'data' / 'rules' / 'insurance_compliance_rules.txt'
SAMPLE_DOC_TXT = REPO_ROOT / 'data' / 'sample_documents' / 'sample_claim.txt'
CHROMA_DIR = REPO_ROOT / 'chroma_db'
HISTORY_FILE = REPO_ROOT / 'audit_history.json'
CHROMA_RULE_COLLECTION = 'compliance_rules'

RULEBOOK_PATH, SAMPLE_DOC_TXT, CHROMA_DIR, HISTORY_FILE

## 3) Document Ingestion (TXT + PDF)

In [ ]:
from typing import Optional

def extract_txt_text(txt_path: Path) -> str:
    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def extract_pdf_text(pdf_path: Path, max_pages: Optional[int] = 12) -> str:
    import fitz  # PyMuPDF
    doc = fitz.open(str(pdf_path))
    pages = doc if max_pages is None else doc[:max_pages]
    return '\n'.join(page.get_text() for page in pages)

def extract_document_text(path: Path, max_pdf_pages: int = 12) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    ext = path.suffix.lower()
    if ext == '.txt':
        return extract_txt_text(path)
    if ext == '.pdf':
        return extract_pdf_text(path, max_pages=max_pdf_pages)
    raise ValueError(f'Unsupported document type: {ext}. Use .txt or .pdf')

sample_text = extract_document_text(SAMPLE_DOC_TXT)
logger.info(f'Sample document loaded: {len(sample_text)} chars')
sample_text[:400]

## 4) Compliance Rulebook

In [ ]:
def load_rulebook_lines(rulebook_path: Path) -> list[str]:
    rulebook_path = Path(rulebook_path)
    if not rulebook_path.exists():
        raise FileNotFoundError(f'Rulebook not found: {rulebook_path}')
    rules = []
    with open(rulebook_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('Insurance Claim Compliance Rules'):
                continue
            if line.startswith('Rule '):
                continue
            rules.append(line)
    return rules

rules = load_rulebook_lines(RULEBOOK_PATH)
logger.info(f'Loaded {len(rules)} rule lines')
rules[:5]

## 5) RAG Pipeline (ChromaDB + sentence-transformers)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(embedding_model_name)

def embed_texts(texts: list[str]) -> list[list[float]]:
    vecs = embedding_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    return vecs.tolist()

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(name=CHROMA_RULE_COLLECTION)

def upsert_rules_to_chroma(rule_texts: list[str], batch_size: int = 64):
    if not rule_texts:
        return
    try:
        existing = collection.count()
    except Exception:
        existing = 0
    if existing >= len(rule_texts):
        logger.info('Chroma already populated; skipping upsert.')
        return
    ids = [f'rule_{i:06d}' for i in range(len(rule_texts))]
    logger.info(f'Upserting {len(rule_texts)} rules into Chroma...')
    for start in range(0, len(rule_texts), batch_size):
        batch = rule_texts[start:start+batch_size]
        batch_ids = ids[start:start+batch_size]
        batch_embeddings = embed_texts(batch)
        collection.upsert(ids=batch_ids, documents=batch, embeddings=batch_embeddings)

def retrieve_top_k_rules(query_text: str, top_k: int = 5) -> list[str]:
    q_emb = embed_texts([query_text])[0]
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    docs = (results.get('documents') or [[]])[0] or []
    return list(docs)

upsert_rules_to_chroma(rules)
retrieved_rules = retrieve_top_k_rules(sample_text[:6000], top_k=5)
retrieved_rules

## 6) Prompt Engineering (STRICT JSON)

In [ ]:
STRICT_JSON_SCHEMA = {
    'summary': '',
    'compliance_score': 0,
    'risk_score': 0,
    'risk_level': '',
    'risk_reasoning': '',
    'violations': [],
    'recommendations': [],
    'confidence_score': 0
}

def build_strict_compliance_prompt(rules: list[str], document_text: str) -> str:
    rules_block = '\n'.join([f'- {r}' for r in rules])
    json_example = json.dumps(STRICT_JSON_SCHEMA, ensure_ascii=False, indent=2)
    return (
        'You are an expert insurance compliance auditor.\n\n'
        'Review the following insurance claim auditing document against the compliance rulebook.\n\n'
        f'Rules (context):\n{rules_block}\n\n'
        f'Document to audit:\n{document_text}\n\n'
        'Return ONLY valid JSON with EXACTLY these keys and value types: ' + json_example + '\n\n'
        'Hard requirements:\n'
        '- Output must be a single JSON object (no markdown, no code fences).\n'
        '- compliance_score, risk_score, confidence_score must be integers (0-100).\n'
        '- risk_level must be one of: LOW, MEDIUM, HIGH, CRITICAL.\n'
        '- violations and recommendations must be arrays of strings.\n'
        '- If unknown, use safe defaults and explain briefly in risk_reasoning.\n'
    )

prompt = build_strict_compliance_prompt(retrieved_rules, sample_text[:4000])
prompt[:700]

## 7) AI Compliance Agent (PydanticAI + vLLM/OpenAI-compatible endpoint)

In [ ]:
import requests
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

def verify_vllm_endpoint(base_url: str, api_key: str, timeout_s: float = 10.0) -> dict:
    """Best-effort validation of /models and a minimal chat-completions call."""
    headers = {'Authorization': f'Bearer {api_key}'}
    probe_urls = [f'{base_url}/models', f'{base_url}/chat/completions']
    out = {'ok': False, 'probe': {}}
    for url in probe_urls:
        try:
            if url.endswith('/models'):
                r = requests.get(url, headers=headers, timeout=timeout_s)
                out['probe']['models'] = {'status': r.status_code, 'text': r.text[:200]}
            else:
                payload = {
                    'model': MODEL_NAME,
                    'messages': [
                        {'role': 'system', 'content': 'Return JSON only.'},
                        {'role': 'user', 'content': 'ping'}
                    ],
                    'temperature': 0.0,
                    'max_tokens': 16,
                }
                r = requests.post(url, headers=headers, json=payload, timeout=timeout_s)
                out['probe']['chat'] = {'status': r.status_code, 'text': r.text[:200]}
            out['ok'] = True
        except Exception as e:
            out['probe'][url] = {'error': str(e)}
    return out

endpoint_check = verify_vllm_endpoint(AMD_BASE_URL, AMD_API_KEY)
logger.info(f'vLLM endpoint check: {endpoint_check}')
endpoint_check

provider = OpenAIProvider(base_url=AMD_BASE_URL, api_key=AMD_API_KEY)
model = OpenAIModel(MODEL_NAME, provider=provider)
agent = Agent(model=model)
logger.info('PydanticAI agent initialized.')

## 8) JSON Parsing Layer (Safe + Fallback)

In [ ]:
def safe_extract_json_object(text: str) -> Optional[dict[str, Any]]:
    text = (text or '').strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = text[start:end+1]
    try:
        return json.loads(candidate)
    except Exception:
        return None

def parse_ai_response_to_dict(ai_response: str) -> dict[str, Any]:
    fallback = {
        'summary': 'Failed to parse AI response',
        'compliance_score': 0,
        'risk_score': 0,
        'risk_level': 'UNKNOWN',
        'risk_reasoning': '',
        'violations': [],
        'recommendations': [],
        'confidence_score': 0,
    }
    parsed = safe_extract_json_object(ai_response)
    if not isinstance(parsed, dict):
        logger.warning('Could not extract JSON; using fallback.')
        return fallback

    def as_int(v, default=0):
        try:
            return int(v)
        except Exception:
            return default

    out = {**fallback, **parsed}
    out['compliance_score'] = as_int(out.get('compliance_score'), 0)
    out['risk_score'] = as_int(out.get('risk_score'), 0)
    out['confidence_score'] = as_int(out.get('confidence_score'), 0)

    for key in ['violations', 'recommendations']:
        val = out.get(key, [])
        if isinstance(val, list):
            out[key] = [str(x) for x in val]
        else:
            out[key] = []

    rl = str(out.get('risk_level', 'UNKNOWN')).upper().strip()
    allowed = {'LOW', 'MEDIUM', 'HIGH', 'CRITICAL', 'UNKNOWN'}
    out['risk_level'] = rl if rl in allowed else 'UNKNOWN'
    return out


## 9) Audit Report Model

In [ ]:
from pydantic import BaseModel, Field

class AuditReport(BaseModel):
    summary: str
    compliance_score: int
    risk_score: int
    risk_level: str
    risk_reasoning: str
    retrieved_rules: list[str] = Field(default_factory=list)
    violations: list[str] = Field(default_factory=list)
    recommendations: list[str] = Field(default_factory=list)
    confidence_score: int

AuditReport.model_json_schema()

## 10) Compliance Orchestrator

In [ ]:
import asyncio

def load_history(path: Path) -> list[dict]:
    if not path.exists():
        return []
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return []

def save_history(path: Path, record: dict) -> None:
    history = load_history(path)
    history.append(record)
    path.write_text(json.dumps(history, indent=2, ensure_ascii=False), encoding='utf-8')

async def run_qwen_compliance(prompt: str) -> str:
    result = await agent.run(prompt)
    return str(result.output)

def create_audit_report(ai_dict: dict[str, Any], retrieved_rules: list[str]) -> AuditReport:
    data = dict(ai_dict)
    data['retrieved_rules'] = retrieved_rules
    return AuditReport(**data)

async def audit_document_pipeline(document_path: Path, top_k_rules: int = 5, max_pdf_pages: int = 12) -> AuditReport:
    logger.info('STEP 1: Loading document')
    doc_text = extract_document_text(document_path, max_pdf_pages=max_pdf_pages)
    doc_text_rag = doc_text[:12000]

    logger.info('STEP 2: Retrieving rules')
    retrieved = retrieve_top_k_rules(doc_text_rag, top_k=top_k_rules)

    logger.info('STEP 3: Building prompt')
    prompt = build_strict_compliance_prompt(retrieved, doc_text_rag)

    logger.info('STEP 4: Running AI Compliance Agent')
    ai_raw = await run_qwen_compliance(prompt)

    logger.info('STEP 5: Parsing AI Response')
    ai_dict = parse_ai_response_to_dict(ai_raw)

    logger.info('STEP 6: Creating Audit Report')
    return create_audit_report(ai_dict, retrieved)

def report_to_record(report: AuditReport, source_path: Path) -> dict:
    return {
        'timestamp': datetime.utcnow().isoformat() + 'Z',
        'source': str(source_path),
        'report': report.model_dump(),
    }


## 11) Report Generation (formatted)

In [ ]:
def format_audit_report(report: AuditReport) -> str:
    lines = []
    lines.append('AI COMPLIANCE AUDIT REPORT')
    lines.append('=' * 32)
    lines.append(f'Compliance Score : {report.compliance_score}%')
    lines.append(f'Risk Score       : {report.risk_score}')
    lines.append(f'Risk Level       : {report.risk_level}')
    lines.append('')
    lines.append('SUMMARY')
    lines.append('-' * 16)
    lines.append(report.summary)
    lines.append('')
    lines.append('RISK REASONING')
    lines.append('-' * 16)
    lines.append(report.risk_reasoning)
    lines.append('')
    lines.append('RETRIEVED RULES')
    lines.append('-' * 18)
    for r in report.retrieved_rules:
        lines.append(f'✓ {r}')
    lines.append('')
    lines.append('VIOLATIONS')
    lines.append('-' * 11)
    if report.violations:
        for v in report.violations:
            lines.append(f'✗ {v}')
    else:
        lines.append('None')
    lines.append('')
    lines.append('RECOMMENDATIONS')
    lines.append('-' * 17)
    if report.recommendations:
        for rec in report.recommendations:
            lines.append(f'• {rec}')
    else:
        lines.append('None')
    lines.append('')
    lines.append(f'Confidence Score : {report.confidence_score}%')
    return '\n'.join(lines)


## 12) Audit History (save/load + table)

In [ ]:
import pandas as pd

def history_to_dataframe(history_records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in history_records:
        r = rec.get('report', {})
        rows.append({
            'timestamp': rec.get('timestamp'),
            'source': rec.get('source'),
            'compliance_score': r.get('compliance_score'),
            'risk_score': r.get('risk_score'),
            'risk_level': r.get('risk_level'),
            'confidence_score': r.get('confidence_score'),
        })
    return pd.DataFrame(rows)

hist = load_history(HISTORY_FILE)
df = history_to_dataframe(hist)
df.tail(10) if len(df) else df

## 13) Demo (end-to-end audit on sample claim)

In [ ]:
async def demo():
    report = await audit_document_pipeline(SAMPLE_DOC_TXT, top_k_rules=5)
    record = report_to_record(report, SAMPLE_DOC_TXT)
    save_history(HISTORY_FILE, record)
    print(format_audit_report(report))
    return report

try:
    report = await demo()
except SyntaxError:
    report = asyncio.run(demo())
report

## 14) Visualization (Matplotlib)

In [ ]:
import matplotlib.pyplot as plt

hist = load_history(HISTORY_FILE)
df = history_to_dataframe(hist)
if len(df) == 0:
    logger.info('No history yet. Run the demo cell first.')
else:
    df_sorted = df.copy()
    df_sorted['timestamp'] = pd.to_datetime(df_sorted['timestamp'], errors='coerce')
    df_sorted = df_sorted.sort_values('timestamp')
    x = list(range(len(df_sorted)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(x, df_sorted['compliance_score'], marker='o')
    axes[0].set_title('Compliance Score Trend')
    axes[0].set_xlabel('Audit Index')
    axes[0].set_ylabel('Compliance Score (%)')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x, df_sorted['risk_score'], marker='o', color='orange')
    axes[1].set_title('Risk Score Trend')
    axes[1].set_xlabel('Audit Index')
    axes[1].set_ylabel('Risk Score')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    df_sorted['risk_level'].value_counts().plot(kind='bar')
    plt.title('Risk Level Distribution')
    plt.xlabel('Risk Level')
    plt.ylabel('Count')
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 15) Production Enhancements (what to notice)

In [ ]:
print('✅ Notebook complete. Run cells top-to-bottom for a full compliance audit.')